<a href="https://colab.research.google.com/github/EvansAmarh/CausalMedia-GH/blob/main/casualmediagh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import random
from glob import glob

In [ ]:
data_path = "/content/drive/MyDrive/ednet/"
files = glob(data_path + "*.csv")
print("Total files:", len(files))

Total files: 23478


In [ ]:
subset_files = random.sample(files, 20000)
print("Subset size:", len(subset_files))

Subset size: 20000


In [ ]:
sample_file = subset_files[0]

df = pd.read_csv(sample_file)

df.head()

,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1547818140748,1,q5758,a,27000


In [ ]:
print(df.columns)

Index(['timestamp', 'solving_id', 'question_id', 'user_answer',
       'elapsed_time'],
      dtype='object')


In [ ]:
all_students = []

In [ ]:
all_students = []

for i, file in enumerate(subset_files):

    try:
        df = pd.read_csv(file)

        interactions = len(df)

        if interactions >= 15:
            all_students.append(df)

        # Show progress every 1000 files
        if (i + 1) % 1000 == 0:
            print(f"Processed {i+1} files")

    except:
        continue

Processed 1000 files
Processed 2000 files
Processed 3000 files
Processed 4000 files
Processed 5000 files
Processed 6000 files
Processed 7000 files
Processed 8000 files
Processed 9000 files
Processed 10000 files
Processed 11000 files
Processed 12000 files
Processed 13000 files
Processed 14000 files
Processed 15000 files
Processed 16000 files
Processed 17000 files
Processed 18000 files
Processed 19000 files
Processed 20000 files


In [ ]:
print("Valid students:", len(all_students))

Valid students: 11223


In [ ]:
sampled_students = random.sample(all_students, 10000)
print("Sampled students:", len(sampled_students))

Sampled students: 10000


In [ ]:
combined_df = pd.concat(sampled_students, ignore_index=True)
print(combined_df.shape)

(7296196, 5)


In [ ]:
combined_df = combined_df.dropna()
print(combined_df.shape)

(7292545, 5)


In [ ]:
combined_df = combined_df.drop_duplicates()
print(combined_df.shape)

(7279353, 5)


In [ ]:
avg_time = combined_df.groupby("solving_id")["elapsed_time"].mean()

In [ ]:
interaction_counts = combined_df.groupby("solving_id").size()

In [ ]:
combined_df["avg_time_spent"] = combined_df["solving_id"].map(avg_time)

combined_df["total_interactions"] = combined_df["solving_id"].map(interaction_counts)
combined_df[["solving_id", "avg_time_spent", "total_interactions"]].head()

,solving_id,avg_time_spent,total_interactions
0,1,27746.023472,10012
1,2,23072.199400,10000
2,3,24490.548255,9999
3,4,23118.923485,9998
4,5,24397.345471,16450


In [ ]:
combined_df["engagement_score"] = (
    combined_df["avg_time_spent"] *
    combined_df["total_interactions"]
)
combined_df[["engagement_score"]].head()


,engagement_score
0,277793187.0
1,230721994.0
2,244880992.0
3,231142997.0
4,401336333.0


In [ ]:
threshold = combined_df["engagement_score"].quantile(0.75)

combined_df["engagement_level"] = np.where(
    combined_df["engagement_score"] >= threshold,
    1,
    0
)
combined_df["engagement_level"].value_counts()

,count
engagement_level,
0,5459490
1,1819863


In [ ]:
question_counts = combined_df.groupby("solving_id")["question_id"].count()

combined_df["performance_gain"] = combined_df["solving_id"].map(question_counts)
combined_df[["solving_id", "performance_gain"]].head()

,solving_id,performance_gain
0,1,10012
1,2,10000
2,3,9999
3,4,9998
4,5,16450


In [ ]:
combined_df["timestamp"] = pd.to_datetime(combined_df["timestamp"])

In [ ]:
combined_df["hour"] = combined_df["timestamp"].dt.hour
combined_df[["timestamp", "hour"]].head()

,timestamp,hour
0,1970-01-01 00:26:06.141225448,0
1,1970-01-01 00:26:06.141249751,0
2,1970-01-01 00:26:06.141270607,0
3,1970-01-01 00:26:06.141287860,0
4,1970-01-01 00:26:06.141316249,0


In [ ]:
combined_df["day"] = combined_df["timestamp"].dt.day
combined_df[["timestamp", "day"]].head()

,timestamp,day
0,1970-01-01 00:26:06.141225448,1
1,1970-01-01 00:26:06.141249751,1
2,1970-01-01 00:26:06.141270607,1
3,1970-01-01 00:26:06.141287860,1
4,1970-01-01 00:26:06.141316249,1


In [ ]:
combined_df.head()


,timestamp,solving_id,question_id,user_answer,elapsed_time,avg_time_spent,total_interactions,engagement_score,engagement_level,performance_gain,hour,day
0,1970-01-01 00:26:06.141225448,1,q5747,d,13000,27746.023472,10012,277793187.0,1,10012,0,1
1,1970-01-01 00:26:06.141249751,2,q4344,c,20000,23072.199400,10000,230721994.0,1,10000,0,1
2,1970-01-01 00:26:06.141270607,3,q5197,a,19000,24490.548255,9999,244880992.0,1,9999,0,1
3,1970-01-01 00:26:06.141287860,4,q5019,b,15000,23118.923485,9998,231142997.0,1,9998,0,1
4,1970-01-01 00:26:06.141316249,5,q5647,b,26000,24397.345471,16450,401336333.0,1,16450,0,1


In [ ]:
print(combined_df.shape)

(7279353, 12)


In [ ]:
output_path = "/content/drive/MyDrive/clean_ednet.csv"

combined_df.to_csv(output_path, index=False)